# 🔎 Beyond Filenames — Personal File Retrieval Benchmark
**Question:** If I forget a file's name but remember what was inside it, can a computer still find it?

This notebook builds a **simulated messy laptop** (600+ real Word, PDF, text and screenshot files with useless names, plus up to 5,000 unrelated files), then compares **7 search methods** on **~1,900 queries** written three ways:
- **Exact words** – using the words inside the file
- **From memory** – describing it the way people actually remember
- **Memory + clue** – memory plus a remembered detail ("a PDF from March")

👉 **Just click `Runtime → Run all`.** It takes about 10–20 minutes. (Faster with `Runtime → Change runtime type → T4 GPU`.)

## 1. Install tools

In [ ]:
!pip -q install rank-bm25 python-docx pymupdf pytesseract sentence-transformers
!apt-get -qq install -y tesseract-ocr > /dev/null
print('✅ Installed')

## 2. Create the experiment code (modules)

In [ ]:
import os
os.makedirs('recall_bench', exist_ok=True)
open('recall_bench/__init__.py','w').close()
print('✅ Folder ready')

In [ ]:
%%writefile recall_bench/config.py
"""All experiment settings in one place. Change these and re-run."""
SEED = 42
WORK_DIR = "fake_laptop"          # where the simulated files are written
OUT_DIR = "results"               # tables + charts go here

# Corpus sizes to test. "Filler" = unrelated files (emails/posts from the public
# 20 Newsgroups dataset) that make the laptop bigger and the search harder.
FILLER_SIZES = [0, 1000, 3000, 5000]

INCLUDE_SCREENSHOTS = True        # make PNG "screenshots" (needs OCR to be searchable)
MEANINGFUL_NAME_RATE = 0.15       # % of files that get a sensible name (realism)

TEXT_MODELS = {                   # display name -> (HF model, query prefix)
    "Semantic (MiniLM)": ("sentence-transformers/all-MiniLM-L6-v2", ""),
    "Semantic (BGE-small)": ("BAAI/bge-small-en-v1.5",
                             "Represent this sentence for searching relevant passages: "),
}
HYBRID_BASE = "Semantic (BGE-small)"   # semantic model used inside the hybrid


In [ ]:
%%writefile recall_bench/data_gen.py
"""Module 1 - builds a simulated messy laptop: real .docx/.pdf/.txt/.png files
with useless names, plus ground-truth queries written two ways:
  exact   - uses the words actually inside the file
  memory  - describes the file the way a person remembers it (different words)
  meta    - memory + a remembered clue (file type / month)
"""
import os, random, datetime, shutil, json
from pathlib import Path
from itertools import product

MONTHS = ["January","February","March","April","May","June","July","August",
          "September","October","November","December"]
TYPE_WORDS = {"pdf": "PDF", "docx": "Word file", "txt": "text note", "png": "screenshot"}

# Each category: template text, two slots of (exact phrase, how people remember it),
# exact-query and memory-query templates. {a}/{b} = exact, {A}/{B} = remembered.
CATEGORIES = [
 dict(name="repair invoice", ext=["pdf", "docx"],
  text="INVOICE No. {num}\nDate: {date}\nService: {a} repair - {b}.\nTotal amount: PKR {amount}. Payment received in cash. Thank you for your business.",
  a=[("laptop","my computer"),("smartphone","my mobile"),("printer","the printing machine"),("refrigerator","the fridge"),("air conditioner","the AC unit"),("motorbike","my bike"),("washing machine","the laundry appliance"),("microwave","the food heating oven"),("tablet","my iPad-type device"),("television","the TV")],
  b=[("screen replacement","they put in a new display"),("battery change","they swapped the power cell"),("motherboard fix","the main board was mended"),("cooling fan cleaning","dust removed from the fan"),("software reinstall","they reset the operating system"),("charging port replacement","the plug-in socket was changed"),("water damage service","it had got wet"),("speaker repair","the sound was not working")],
  exact="{a} repair invoice {b}", memory=["the bill for fixing {A} when {B}","receipt from the shop that mended {A}, {B}"]),
 dict(name="utility bill", ext=["pdf"],
  text="{a} BILL\nCustomer reference {num}\nBilling date {date}\nAmount payable: PKR {amount}\nNote: consumption was higher this cycle due to {b}.",
  a=[("electricity","power"),("gas","heating fuel"),("water","tap supply"),("broadband internet","home wifi"),("mobile postpaid","phone package")],
  b=[("air conditioner usage","running the AC all summer"),("guests staying","relatives visiting us"),("a faulty meter","the reading machine was broken"),("work from home","I stayed home for my job"),("a water leakage","a pipe was dripping"),("extra devices","we plugged in more gadgets")],
  exact="{a} bill {b}", memory=["the {A} charges that went up because {B}","statement saying {A} cost more since {B}"]),
 dict(name="rental agreement", ext=["docx", "pdf"],
  text="RENTAL AGREEMENT\nThis tenancy agreement dated {date} concerns a {a} located in {b}, Karachi.\nMonthly rent PKR {amount}. Security deposit equal to two months. Lease period eleven months.",
  a=[("apartment","flat"),("house","home"),("shop","store"),("office space","workplace"),("single room","bedroom to live in"),("warehouse","storage godown")],
  b=[("Gulshan-e-Iqbal","Gulshan"),("DHA Phase 6","DHA"),("Clifton Block 5","Clifton"),("North Nazimabad","North Nazimabad"),("Bahadurabad","Bahadurabad"),("PECHS","PECHS"),("Saddar","Saddar")],
  exact="rental agreement {a} {b}", memory=["the lease contract for the {A} in {B}","papers for renting that {A} near {B}"]),
 dict(name="lecture notes", ext=["docx", "pdf", "txt"],
  text="Course notes - {b} preparation\nTopic: {a}.\nKey definitions, worked examples and past paper questions are summarised below.\nRevise this topic carefully.",
  a=[("priority inversion","an important task stuck behind a less important one"),("deadlock","processes blocking each other forever"),("database normalization","organising tables to avoid repeated data"),("gradient descent","slowly adjusting weights to reduce error"),("binary search","finding an item by halving the list again and again"),("TCP three-way handshake","how two computers greet before talking"),("recursion","a function that calls itself"),("hash tables","storing values against keys for quick lookup"),("overfitting","a model memorising its training data"),("public key encryption","scrambling messages with two keys"),("cache memory","small fast storage near the processor"),("virtual memory","using disk as extra RAM"),("SQL joins","combining two tables together"),("Big O notation","measuring how quickly an algorithm slows down"),("convolutional neural networks","image recognising AI layers"),("paging","splitting memory into fixed blocks"),("context switching","the CPU swapping between programs"),("load balancing","spreading traffic across servers"),("version control with Git","tracking code changes over time"),("REST APIs","how web services talk to each other")],
  b=[("midterm exam","mid semester test"),("final exam","end of term paper"),("quiz","short class test"),("viva","oral examination")],
  exact="{a} notes {b}", memory=["my study material about {A} for the {B}","the revision sheet explaining {A}, made for the {B}"]),
 dict(name="meeting minutes", ext=["docx", "txt"],
  text="MINUTES OF MEETING\nDate: {date}\nClient: {a} sector account.\nDiscussion: the client requested {b}. Demo to be scheduled next week. Action items assigned to the development team.",
  a=[("textile","clothing manufacturer"),("pharmaceutical","medicine company"),("agriculture","farming business"),("logistics","shipping firm"),("retail","shop chain"),("education","school group"),("restaurant","food business"),("real estate","property dealer")],
  b=[("a sales dashboard","a screen showing how much they sell"),("inventory tracking","keeping count of their stock"),("a payroll module","paying staff salaries"),("an attendance system","recording who comes to work"),("invoice automation","making bills automatically"),("a mobile application","something on phones"),("a customer portal","a website for their buyers"),("data migration","moving their old records over")],
  exact="meeting minutes {a} client {b}", memory=["notes from our call with the {A} about {B}","what the {A} asked for: {B}"]),
 dict(name="travel document", ext=["pdf"],
  text="CONFIRMATION - {a}\nReference {num}, issued {date}.\nDestination: {b}.\nPlease carry your passport and this confirmation.",
  a=[("flight ticket","plane booking"),("hotel reservation","place to stay"),("visa appointment","embassy slot"),("travel insurance","cover for the trip")],
  b=[("Dubai","the UAE"),("Istanbul","Turkey"),("London","the UK"),("Kuala Lumpur","Malaysia"),("Jeddah","Saudi Arabia"),("Doha","Qatar"),("Baku","Azerbaijan"),("Berlin","Germany"),("Toronto","Canada"),("Skardu","the northern mountains")],
  exact="{a} {b} confirmation", memory=["my {A} for the trip to {B}","booking proof for {B}, the {A}"]),
 dict(name="recipe", ext=["docx", "txt"],
  text="Recipe: {a}\nSource: {b}.\nIngredients and step-by-step method written below. Serves four people.",
  a=[("chicken karahi","spicy chicken cooked in a wok"),("beef nihari","slow cooked breakfast meat stew"),("chocolate fudge cake","sweet brown dessert"),("chicken biryani","layered rice and meat dish"),("daal chawal","lentils with plain rice"),("aloo paratha","potato stuffed flatbread"),("kheer","milky rice pudding"),("haleem","thick wheat and meat porridge"),("shami kebab","minced patties"),("gajar ka halwa","carrot sweet dish")],
  b=[("my mother's handwritten notes","how mum makes it"),("a YouTube cooking channel","an online video"),("my grandmother","how nani makes it"),("a restaurant chef","a cook at an eatery")],
  exact="{a} recipe {b}", memory=["how to make {A}, {B}","cooking steps for {A} the way {B}"]),
 dict(name="cover letter", ext=["docx", "pdf"],
  text="Dear Hiring Manager,\nI am applying for the {a} position at your {b}. My university projects and work experience have prepared me well for this role.\nSincerely,",
  a=[("software engineer","coding job"),("data analyst","job working with numbers and charts"),("ERP consultant","business systems advisor role"),("teaching assistant","helping a professor"),("summer internship","trainee placement"),("project manager","team leading role"),("UI designer","making app screens look nice"),("QA tester","finding bugs role")],
  b=[("bank","financial firm"),("startup","new small company"),("university","campus"),("telecom company","mobile network operator"),("multinational corporation","global firm")],
  exact="cover letter {a} {b}", memory=["my application for the {A} at a {B}","the job letter I wrote for a {A} at a {B}"]),
 dict(name="warranty card", ext=["pdf", "png"],
  text="WARRANTY CERTIFICATE\nProduct: {a}\nSerial {num}\nCoverage period: {b} from date of purchase {date}.\nKeep this card safe for claims.",
  a=[("washing machine","laundry appliance"),("microwave oven","food heating box"),("smart LED TV","big screen"),("water dispenser","cooler for drinking"),("split air conditioner","wall AC"),("gaming laptop","computer for games"),("electric kettle","tea water boiler"),("vacuum cleaner","floor dust sucker")],
  b=[("one year","12 months"),("two years","24 months"),("six months","half a year"),("three years","36 months")],
  exact="{a} warranty {b}", memory=["guarantee paper for the {A}, {B} cover","proof the {A} is covered for {B}"]),
 dict(name="error screenshot", ext=["png"],
  text="{a}\nError: {b}\nPlease try again later or contact support.",
  a=[("Online Banking Login","my money app"),("Outlook Mail","the email website"),("Student Portal","the university website"),("WhatsApp Web","chat app on the computer"),("Windows Update","the computer update"),("Payment Checkout","the online shopping page")],
  b=[("password incorrect","it said my login details were wrong"),("connection timed out","the internet took too long"),("storage full","no space was left"),("access denied","I was not allowed in"),("session expired","it logged me out"),("internal server error 500","the site itself broke")],
  exact="{a} {b}", memory=["picture of {A} when {B}","the capture showing {A} where {B}"]),
 dict(name="programming assignment", ext=["docx", "txt", "pdf"],
  text="Assignment brief\nLanguage: {b}\nTask: implement a {a}. Include comments, test cases and a short report on time complexity.",
  a=[("merge sort","way to put numbers in order by splitting"),("linked list","chain of connected nodes"),("calculator program","tool to add and subtract"),("student management system","records of pupils"),("tic tac toe game","noughts and crosses"),("bank account class","deposit and withdrawal simulation"),("file encryption tool","locking documents with a secret"),("weather API client","fetching the forecast online")],
  b=[("Python","Python"),("Java","Java"),("C++","C++"),("JavaScript","JavaScript"),("SQL","SQL")],
  exact="{b} assignment {a}", memory=["the {B} homework about {A}","coursework in {B}: {A}"]),
 dict(name="event planning", ext=["docx", "txt"],
  text="{b} for the {a}\nPrepared on {date}.\nPlease review and confirm before Friday.",
  a=[("wedding","marriage function"),("birthday party","special day celebration"),("Eid dinner","festival feast"),("office picnic","team outing"),("iftar gathering","breaking the fast together"),("housewarming","new home party")],
  b=[("Guest list","who was invited"),("Budget plan","how much money to spend"),("Shopping list","things to buy"),("Food menu","what dishes to serve")],
  exact="{b} {a}", memory=["{B} for the {A}","the sheet of {B} for that {A}"]),
 dict(name="business report", ext=["docx", "pdf"],
  text="MANAGEMENT REPORT\nSubject: {a} analysis - {b} branch\nReporting date {date}.\nFigures, trends and recommendations follow.",
  a=[("quarterly sales","how much we sold in three months"),("customer complaints","people unhappy with our service"),("employee attendance","who showed up to work"),("website traffic","how many people visited online"),("inventory shrinkage","stock going missing"),("delivery delays","shipments arriving late")],
  b=[("Karachi","Karachi"),("Lahore","Lahore"),("Islamabad","Islamabad"),("Peshawar","Peshawar"),("Multan","Multan"),("Quetta","Quetta")],
  exact="{a} report {b} branch", memory=["the {B} office study on {A}","numbers about {A} for {B}"]),
 dict(name="certificate", ext=["pdf", "png"],
  text="CERTIFICATE OF COMPLETION\nThis certifies successful completion of the course {a}, offered through {b}.\nIssued {date}.",
  a=[("Data Analytics","working with numbers and charts"),("Cloud Computing","renting servers online"),("Cyber Security","protecting computers from hackers"),("Digital Marketing","online advertising"),("Project Management","running projects properly"),("Public Speaking","talking in front of people"),("Graphic Design","making visuals"),("Machine Learning","teaching computers from examples")],
  b=[("Coursera","Coursera"),("Google","Google"),("Microsoft","Microsoft"),("Udemy","Udemy"),("LinkedIn Learning","LinkedIn")],
  exact="certificate {a} {b}", memory=["proof I finished the {B} course on {A}","the award from {B} for {A}"]),
]

JUNK = ["final_final_v{n}", "New Document ({n})", "Screenshot 2026-0{m}-1{d} 11{n:04d}", "IMG_{n:04d}",
        "bas_kar_do_{n}", "untitled{n}", "copy of copy ({n})", "doc{n}", "abc{n}", "draft_{n}_FINAL",
        "scan{n:04d}", "WhatsApp Document 2026-0{m}-2{d}", "ok_{n}", "lol{n}", "temp{n}", "xyz_final{n}"]


def _junk_name(rng, i, ext):
    pat = rng.choice(JUNK)
    if ext == "png" and rng.random() < 0.6:
        pat = rng.choice(["Screenshot 2026-0{m}-1{d} 11{n:04d}", "IMG_{n:04d}"])
    return pat.format(n=i, m=rng.randint(1, 9), d=rng.randint(0, 9)) + "." + ext


def _write(path, text, ext):
    if ext == "txt":
        path.write_text(text, encoding="utf-8")
    elif ext == "docx":
        import docx
        d = docx.Document()
        for line in text.split("\n"):
            d.add_paragraph(line)
        d.save(path)
    elif ext == "pdf":
        import pymupdf
        doc = pymupdf.open()
        page = doc.new_page()
        page.insert_textbox(pymupdf.Rect(60, 60, 540, 780), text, fontsize=11)
        doc.save(path)
    elif ext == "png":
        from PIL import Image, ImageDraw, ImageFont
        from matplotlib import font_manager
        font = ImageFont.truetype(font_manager.findfont("DejaVu Sans"), 28)
        img = Image.new("RGB", (1200, 420), "white")
        dr = ImageDraw.Draw(img)
        dr.rectangle([20, 20, 1180, 400], outline=(200, 30, 30), width=4)
        for k, line in enumerate(text.split("\n")):
            dr.text((50, 60 + k * 70), line, fill="black", font=font)
        img.save(path)


def generate(work_dir, seed=42, include_screenshots=True, meaningful_rate=0.15, max_filler=5000):
    rng = random.Random(seed)
    root = Path(work_dir)
    if root.exists():
        shutil.rmtree(root)
    folders = ["Desktop", "Downloads", "Documents", "Documents/New folder", "Documents/New folder (2)",
               "Pictures", "WhatsApp", "University", "Office/old"]
    for f in folders:
        (root / f).mkdir(parents=True, exist_ok=True)

    files, queries, i = [], [], 0
    for cat in CATEGORIES:
        exts = [e for e in cat["ext"] if include_screenshots or e != "png"] or ["pdf"]
        for (a, A), (b, B) in product(cat["a"], cat["b"]):
            i += 1
            ext = rng.choice(exts)
            day = datetime.date(2025, 1, 1) + datetime.timedelta(days=rng.randint(0, 600))
            text = cat["text"].format(a=a, b=b, num=rng.randint(1000, 9999), date=day.strftime("%d/%m/%Y"),
                                      amount=f"{rng.randint(2, 90) * 500:,}")
            if rng.random() < meaningful_rate:
                name = f"{cat['name'].replace(' ', '_')}_{i}.{ext}"
            else:
                name = _junk_name(rng, i, ext)
            folder = "Pictures" if ext == "png" and rng.random() < 0.5 else rng.choice(folders)
            path = root / folder / name
            _write(path, text, ext)
            ts = datetime.datetime(day.year, day.month, day.day, 12).timestamp()
            os.utime(path, (ts, ts))
            fid = len(files)
            files.append(dict(id=fid, path=str(path), name=name, ext=ext, category=cat["name"],
                              month=day.month, filler=False))
            memory = rng.choice(cat["memory"]).format(A=A, B=B)
            queries += [
                dict(target=fid, qtype="exact", query=cat["exact"].format(a=a, b=b)),
                dict(target=fid, qtype="memory", query=memory),
                dict(target=fid, qtype="memory+clue",
                     query=f"{memory}. I think it was a {TYPE_WORDS[ext]} from {MONTHS[day.month - 1]}"),
            ]

    # Filler: unrelated real-world text (newsgroup emails) to make the laptop bigger
    filler_texts = _load_filler(max_filler, rng)
    for k, t in enumerate(filler_texts):
        ext = "txt"
        name = rng.choice(["message_{n}", "note{n}", "export_{n}", "old_mail_{n}", "file{n}"]).format(n=k) + ".txt"
        path = root / rng.choice(folders) / name
        path.write_text(t, encoding="utf-8")
        day = datetime.date(2024, 1, 1) + datetime.timedelta(days=rng.randint(0, 1000))
        ts = datetime.datetime(day.year, day.month, day.day, 12).timestamp()
        os.utime(path, (ts, ts))
        files.append(dict(id=len(files), path=str(path), name=name, ext=ext, category="filler",
                          month=day.month, filler=True))
    return files, queries


def _load_filler(n, rng):
    if n <= 0:
        return []
    try:
        from sklearn.datasets import fetch_20newsgroups
        data = fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes")).data
        data = [d.strip()[:1500] for d in data if len(d.strip()) > 200]
        rng.shuffle(data)
        print(f"Loaded {min(n, len(data))} filler documents from 20 Newsgroups")
        return data[:n]
    except Exception as e:
        print(f"[warn] Could not load filler documents ({e}). Running without filler.")
        return []


In [ ]:
%%writefile recall_bench/extract.py
"""Module 2 - reads text back out of the files, exactly like a desktop indexer would."""
from pathlib import Path


def ocr_ready():
    try:
        import pytesseract
        pytesseract.get_tesseract_version()
        return True
    except Exception:
        return False


def read_text(path, use_ocr):
    p = Path(path)
    ext = p.suffix.lower()
    try:
        if ext == ".txt":
            return p.read_text(encoding="utf-8", errors="ignore")
        if ext == ".docx":
            import docx
            return "\n".join(par.text for par in docx.Document(p).paragraphs)
        if ext == ".pdf":
            import pymupdf
            with pymupdf.open(p) as d:
                return "\n".join(pg.get_text() for pg in d)
        if ext == ".png" and use_ocr:
            import pytesseract
            from PIL import Image
            return pytesseract.image_to_string(Image.open(p))
    except Exception as e:
        print(f"[skip] {p.name}: {e}")
    return ""


In [ ]:
%%writefile recall_bench/methods.py
"""Module 3 - the search methods being compared. Each returns a ranked list of file ids."""
import re, time
import numpy as np
from rank_bm25 import BM25Okapi

MONTHS = ["january","february","march","april","may","june","july","august",
          "september","october","november","december"]
TYPE_HINTS = {"pdf": "pdf", "word": "docx", "screenshot": "png", "picture": "png", "text note": "txt"}


def tok(t):
    return re.findall(r"[a-z0-9]+", t.lower())


class Filename:
    name = "Filename search"
    def fit(self, files, texts):
        self.bm = BM25Okapi([tok(f["name"]) or ["_"] for f in files])
    def scores(self, q):
        return self.bm.get_scores(tok(q))


class Keyword:
    name = "Keyword (BM25)"
    def fit(self, files, texts):
        self.bm = BM25Okapi([tok(t) or ["_"] for t in texts])
    def scores(self, q):
        return self.bm.get_scores(tok(q))


class Semantic:
    """Embeddings are computed once for the full corpus and sliced per corpus size."""
    def __init__(self, name, model_name, query_prefix, all_texts):
        from sentence_transformers import SentenceTransformer
        self.name, self.prefix = name, query_prefix
        self.model = SentenceTransformer(model_name)
        t0 = time.time()
        self.all_emb = self.model.encode(all_texts, batch_size=64, normalize_embeddings=True,
                                         show_progress_bar=True)
        self.index_seconds_full = time.time() - t0
        self.n_full = len(all_texts)
    def fit(self, files, texts):
        self.emb = self.all_emb[[f["id"] for f in files]]
    def encode_queries(self, qs):
        return self.model.encode([self.prefix + q for q in qs], batch_size=64, normalize_embeddings=True)
    def scores_from_vec(self, qv):
        return self.emb @ qv


def rrf(score_lists, k=60):
    """Reciprocal Rank Fusion: combine rankings, not raw scores (robust to scale differences)."""
    total = np.zeros(len(score_lists[0]))
    for s in score_lists:
        ranks = np.empty(len(s)); ranks[np.argsort(-s)] = np.arange(1, len(s) + 1)
        total += 1.0 / (k + ranks)
    return total


def naive_blend(kw, sem, alpha=0.5):
    kw = kw / kw.max() if kw.max() > 0 else kw
    sem = (sem - sem.min()) / (sem.max() - sem.min() + 1e-9)
    return alpha * sem + (1 - alpha) * kw


def clue_mask(q, files):
    """Turn remembered clues in the query ('a PDF from March') into a filter."""
    ql = q.lower()
    mask = np.ones(len(files), bool)
    for m_i, m in enumerate(MONTHS, start=1):
        if re.search(rf"\b{m}\b", ql):
            mask &= np.array([f["month"] == m_i for f in files])
    for hint, ext in TYPE_HINTS.items():
        if hint in ql:
            mask &= np.array([f["ext"] == ext for f in files])
    return mask if mask.any() else np.ones(len(files), bool)


In [ ]:
%%writefile recall_bench/metrics.py
"""Module 4 - standard information-retrieval metrics."""
import numpy as np


def rank_of(scores, target_pos):
    order = np.argsort(-scores, kind="stable")
    return int(np.where(order == target_pos)[0][0]) + 1


def summarise(ranks):
    r = np.array(ranks)
    return dict(hit1=round(100 * np.mean(r == 1), 1), hit5=round(100 * np.mean(r <= 5), 1),
                hit10=round(100 * np.mean(r <= 10), 1), mrr=round(float(np.mean(1 / r)), 3))


def content_overlap(q, text, stop):
    import re
    qw = {w for w in re.findall(r"[a-z0-9]+", q.lower()) if w not in stop and len(w) > 2}
    tw = set(re.findall(r"[a-z0-9]+", text.lower()))
    return len(qw & tw) / max(len(qw), 1)


In [ ]:
%%writefile recall_bench/plots.py
"""Module 5 - publication-ready charts."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COLORS = {"Filename search": "#9AA5B1", "Keyword (BM25)": "#1F3A5F", "Semantic (MiniLM)": "#6FB7AE",
          "Semantic (BGE-small)": "#2A9D8F", "Hybrid (naive blend)": "#F4A261",
          "Hybrid (RRF)": "#E76F51", "Hybrid (RRF) + clues": "#B5179E"}


def _style(ax):
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)


def main_chart(df, out):
    big = df[df.corpus_size == df.corpus_size.max()]
    methods = list(dict.fromkeys(big.method))
    qtypes = ["exact", "memory", "memory+clue"]
    labels = {"exact": "Exact words", "memory": "From memory", "memory+clue": "Memory + clue"}
    fig, ax = plt.subplots(figsize=(11, 4.8))
    w = 0.8 / len(methods); x = np.arange(len(qtypes))
    for i, m in enumerate(methods):
        vals = [big[(big.method == m) & (big.qtype == q)].hit5.values[0] for q in qtypes]
        bars = ax.bar(x + (i - len(methods) / 2 + 0.5) * w, vals, w, label=m, color=COLORS.get(m))
        for b in bars:
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{b.get_height():.0f}",
                    ha="center", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels([labels[q] for q in qtypes])
    ax.set_ylabel("Correct file in top 5 (%)"); ax.set_ylim(0, 112); _style(ax)
    ax.set_title(f"How often each method finds the file ({int(big.corpus_size.iloc[0]):,} files)",
                 loc="left", fontweight="bold", color="#1F3A5F")
    ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches="tight"); plt.close()


def scale_chart(df, out, qtype="memory"):
    sub = df[df.qtype == qtype]
    fig, ax = plt.subplots(figsize=(10, 4.4))
    for m in dict.fromkeys(sub.method):
        s = sub[sub.method == m].sort_values("corpus_size")
        ax.plot(s.corpus_size, s.hit1, marker="o", label=m, color=COLORS.get(m), lw=2)
    ax.set_xlabel("Number of files on the laptop"); ax.set_ylabel("Correct file ranked #1 (%)")
    ax.set_title("Does search get worse as the laptop fills up? (memory-style queries)", loc="left",
                 fontweight="bold", color="#1F3A5F")
    ax.legend(frameon=False, fontsize=8); _style(ax); ax.set_ylim(0, 100)
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches="tight"); plt.close()


def speed_chart(speed, out):
    s = pd.DataFrame(speed)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, col, title in [(axes[0], "index_seconds", "Time to index all files (s)"),
                           (axes[1], "ms_per_query", "Time per search (ms)")]:
        ax.barh(s.method, s[col], color=[COLORS.get(m) for m in s.method])
        for y, v in enumerate(s[col]):
            ax.text(v, y, f" {v:.1f}", va="center", fontsize=8)
        ax.set_title(title, loc="left", fontweight="bold", color="#1F3A5F"); _style(ax)
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches="tight"); plt.close()


def overlap_chart(ov, out):
    fig, ax = plt.subplots(figsize=(10, 3.8))
    bins = np.linspace(0, 1, 11)
    ax.hist(ov["exact"], bins=bins, alpha=0.85, label="Exact-word queries", color="#1F3A5F")
    ax.hist(ov["memory"], bins=bins, alpha=0.75, label="Memory-style queries", color="#2A9D8F")
    ax.set_xlabel("Share of query words that actually appear in the file")
    ax.set_ylabel("Number of queries")
    ax.set_title("The 'word gap': people remember meaning, not wording", loc="left",
                 fontweight="bold", color="#1F3A5F")
    ax.legend(frameon=False); _style(ax)
    plt.tight_layout(); plt.savefig(out, dpi=200, bbox_inches="tight"); plt.close()


In [ ]:
%%writefile recall_bench/run_benchmark.py
"""Runs the whole study: generate files -> extract -> index -> search -> score -> chart."""
import json, time, random
from pathlib import Path
import numpy as np
import pandas as pd

from recall_bench import config as C
from recall_bench import data_gen, extract, methods as M, metrics, plots

STOP = set("the a an of to for and or in on at my i me we our is was it that this with by from be "
           "have had when what which how about one where are as think there their they its into".split())


def main():
    out = Path(C.OUT_DIR); out.mkdir(exist_ok=True)
    print("Step 1/5  Building the simulated laptop...")
    files, queries = data_gen.generate(C.WORK_DIR, C.SEED, C.INCLUDE_SCREENSHOTS,
                                       C.MEANINGFUL_NAME_RATE, max(C.FILLER_SIZES))
    targets = [f for f in files if not f["filler"]]
    fillers = [f for f in files if f["filler"]]
    print(f"   {len(targets)} target files, {len(fillers)} filler files, {len(queries)} queries")

    print("Step 2/5  Reading text out of every file (like a desktop indexer)...")
    use_ocr = extract.ocr_ready()
    print(f"   OCR for screenshots: {'ON' if use_ocr else 'OFF (screenshots unsearchable by text)'}")
    t0 = time.time()
    texts = [extract.read_text(f["path"], use_ocr) for f in files]
    extract_s = time.time() - t0

    print("Step 3/5  Loading AI models and embedding all files...")
    semantic = {n: M.Semantic(n, mdl, pre, texts) for n, (mdl, pre) in C.TEXT_MODELS.items()}
    qtexts = [q["query"] for q in queries]
    qvecs, qenc_ms = {}, {}
    for n, s in semantic.items():
        t = time.time(); qvecs[n] = s.encode_queries(qtexts)
        qenc_ms[n] = 1000 * (time.time() - t) / len(qtexts)   # model time per query

    sizes = sorted({len(targets) + k for k in C.FILLER_SIZES if k <= len(fillers)})
    rows, speed, failures = [], [], []
    print("Step 4/5  Searching...")
    for size in sizes:
        corpus = targets + fillers[: size - len(targets)]
        ctexts = [texts[f["id"]] for f in corpus]
        pos = {f["id"]: i for i, f in enumerate(corpus)}
        fn, kw = M.Filename(), M.Keyword()
        t_idx = time.time(); fn.fit(corpus, ctexts); fn_idx = time.time() - t_idx
        t_idx = time.time(); kw.fit(corpus, ctexts); kw_idx = time.time() - t_idx
        for s in semantic.values():
            s.fit(corpus, ctexts)
        ranks = {}; timing = {}
        def add(m, qi, r):
            ranks.setdefault(m, {}).setdefault(queries[qi]["qtype"], []).append(r)
        for qi, q in enumerate(queries):
            tp = pos[q["target"]]
            t = time.perf_counter(); s_fn = fn.scores(q["query"]); timing.setdefault("Filename search", []).append(time.perf_counter() - t)
            t = time.perf_counter(); s_kw = kw.scores(q["query"]); timing.setdefault("Keyword (BM25)", []).append(time.perf_counter() - t)
            add("Filename search", qi, metrics.rank_of(s_fn, tp))
            add("Keyword (BM25)", qi, metrics.rank_of(s_kw, tp))
            sem_scores = {}
            for n, s in semantic.items():
                t = time.perf_counter(); sc = s.scores_from_vec(qvecs[n][qi]); timing.setdefault(n, []).append(time.perf_counter() - t)
                sem_scores[n] = sc
                add(n, qi, metrics.rank_of(sc, tp))
            base = sem_scores[C.HYBRID_BASE]
            add("Hybrid (naive blend)", qi, metrics.rank_of(M.naive_blend(s_kw, base), tp))
            fused = M.rrf([s_kw, base])
            add("Hybrid (RRF)", qi, metrics.rank_of(fused, tp))
            t = time.perf_counter(); mask = M.clue_mask(q["query"], corpus)
            clued = np.where(mask, fused, -1.0); timing.setdefault("Hybrid (RRF) + clues", []).append(time.perf_counter() - t)
            r_clue = metrics.rank_of(clued, tp)
            add("Hybrid (RRF) + clues", qi, r_clue)
            if size == max(sizes) and q["qtype"] == "memory" and r_clue > 5 and len(failures) < 40:
                top = corpus[int(np.argmax(clued))]
                failures.append(dict(query=q["query"], target=files[q["target"]]["name"],
                                     target_rank=r_clue, top_result=top["name"],
                                     top_result_text=texts[top["id"]][:160].replace("\n", " ")))
        for m, by in ranks.items():
            for qt, r in by.items():
                rows.append(dict(corpus_size=size, method=m, qtype=qt, n_queries=len(r), **metrics.summarise(r)))
        if size == max(sizes):
            idx = {"Filename search": fn_idx, "Keyword (BM25)": kw_idx}
            for n, s in semantic.items():
                idx[n] = s.index_seconds_full
            for m in ["Filename search", "Keyword (BM25)", *semantic]:
                speed.append(dict(method=m, index_seconds=round(idx[m] + extract_s, 2),
                                  ms_per_query=round(1000 * np.mean(timing[m]) + qenc_ms.get(m, 0), 3)))
        print(f"   corpus of {size:,} files done")

    print("Step 5/5  Writing tables and charts...")
    df = pd.DataFrame(rows)
    df.to_csv(out / "results.csv", index=False)
    pd.DataFrame(failures).to_csv(out / "failure_examples.csv", index=False)
    ov = {qt: [metrics.content_overlap(q["query"], texts[q["target"]], STOP) for q in queries if q["qtype"] == qt]
          for qt in ["exact", "memory"]}
    zero_gap = 100 * np.mean(np.array(ov["memory"]) == 0)
    plots.main_chart(df, out / "chart_main.png")
    plots.scale_chart(df, out / "chart_scale.png")
    plots.speed_chart(speed, out / "chart_speed.png")
    plots.overlap_chart(ov, out / "chart_word_gap.png")
    pd.DataFrame(speed).to_csv(out / "speed.csv", index=False)

    junk = 100 * np.mean([not (set(M.tok(f["name"])) & set(M.tok(texts[f["id"]]))) for f in targets])
    summary = dict(target_files=len(targets), filler_files=len(fillers), queries=len(queries),
                   largest_corpus=max(sizes), ocr=use_ocr, pct_names_unrelated_to_content=round(junk, 1),
                   pct_memory_queries_zero_word_overlap=round(zero_gap, 1),
                   avg_overlap_exact=round(float(np.mean(ov["exact"])), 2),
                   avg_overlap_memory=round(float(np.mean(ov["memory"])), 2))
    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    big = df[df.corpus_size == max(sizes)].pivot_table(index="method", columns="qtype", values="hit5")
    print("\n=== RESULTS: correct file in top 5 (%), largest corpus ===")
    print(big.round(1).to_string())
    print("\n=== SUMMARY ===")
    print(json.dumps(summary, indent=2))
    return df, summary


if __name__ == "__main__":
    main()


## 3. Run the experiment

In [ ]:
from recall_bench import run_benchmark
df, summary = run_benchmark.main()

## 4. Charts

In [ ]:
from IPython.display import Image, display
for c in ['chart_main','chart_scale','chart_word_gap','chart_speed']:
    display(Image(f'results/{c}.png'))

## 5. Full results table and examples

In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
big = df[df.corpus_size == df.corpus_size.max()]
display(big.pivot_table(index='method', columns='qtype', values=['hit1','hit5']).round(1))
print('\nWhere even the best method failed (useful for the article):')
display(pd.read_csv('results/failure_examples.csv').head(10))

In [ ]:
# Peek at the simulated laptop (good for a screenshot)
import json
from recall_bench import data_gen
files = sorted(os.listdir('fake_laptop/Downloads'))[:15]
print('fake_laptop/Downloads/'); [print('   ', f) for f in files]

## 6. Download everything (charts + tables)

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')